<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Assemble the report</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Collect every chart from notebooks 1&ndash;3 into <strong>one self-contained HTML report</strong> and a matching <strong>data workbook</strong>, then open the report inside TIP.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; landscape analyses after <span style="color: #be0f05; font-weight: 600;">Riccardo Priore</span>, Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Collect every analysis contribution<br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Build the self-contained HTML report<br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Build the data workbook (one sheet per chart)<br/>Step&nbsp;4 &nbsp;&middot;&nbsp; Open the report
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 4 of 4 &mdash; run the four notebooks of this module in order.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Re-running needs <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via <code>epo.tipdata</code>).
            Each notebook writes what the next one reads; notebook&nbsp;4 assembles the report.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

## Step 1 — Collect every analysis contribution

Each earlier notebook left an inline figure fragment plus its data in its output folder, noted in a small manifest. `report_kit.load_contributions` gathers them all and orders them by their `order` number, so the report reads top-to-bottom in a deliberate sequence.

In [9]:
from pathlib import Path
import report_kit

ROOT = Path.cwd()
entries = report_kit.load_contributions(ROOT)
for e in entries:
    print(f"{e['order']:>4}  {e['title']}  ({len(e['sheets'])} sheet(s))")
print(f"\n{len(entries)} contributions.")

 110  Patent families over time  (1 sheet(s))
 120  Leading technology areas (IPC classes)  (1 sheet(s))
 210  Filings by international / regional authority  (1 sheet(s))
 220  WO (PCT) vs EP over time  (1 sheet(s))
 230  National filing trends  (1 sheet(s))
 240  Innovation waves by technology area  (1 sheet(s))
 250  National vs international filing strategy  (1 sheet(s))
 255  Family size & global reach  (1 sheet(s))
 260  Top applicants by patent families  (1 sheet(s))
 270  Applicants by institutional sector  (1 sheet(s))
 280  Grant rate by top applicants  (1 sheet(s))
 290  Most influential organisations (forward citations)  (1 sheet(s))
 310  Technology co-occurrence network  (2 sheet(s))

13 contributions.


## Step 2 — Build the self-contained HTML report (paged + one-page)

We embed **one** copy of plotly.js in the page head, then render every figure as its own
inline `<section>` — no iframes anywhere, which is exactly what lets the report render inside
TIP (Jupyter's `/files/` sandbox blocks the JavaScript an iframe would need).

The report has **two modes, switched by a button in the header** and remembered between visits:

- **Paged** (default): one chart per view, with a numbered step bar, Previous/Next buttons and
  ←/→ arrow keys — the format that works well when presenting live.
- **One page**: every chart stacked in a single scroll — best for scanning or export.

A chart laid out while hidden has no size, so we call `Plotly.Plots.resize` whenever a section
is revealed — without it, paged charts would render blank.

In [10]:
from plotly.offline import get_plotlyjs

REPORT_DIR = ROOT / "4_report"
REPORT_DIR.mkdir(exist_ok=True)
report_path = REPORT_DIR / "antibiotic_resistance_report.html"

# one inline <section> per contribution (no iframes) + one step-bar button carrying its title
sections, steps = [], []
for i, e in enumerate(entries):
    fragment = Path(e["fragment_path"]).read_text(encoding="utf-8")
    note = f'<p class="note">{e["note"]}</p>' if e.get("note") else ""
    sections.append(
        f'<section class="page" id="page{i}" hidden>'
        f'<h2>{e["title"]}</h2>{note}<div class="chart">{fragment}</div></section>')
    steps.append(f'<button class="step" data-go="{i}" title="{e["title"]}">{i + 1}</button>')

# Placeholders (not an f-string): the plotly.js blob and the fragments contain many braces.
TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Antibiotic Resistance &mdash; Patent Landscape</title>
<script>@@PLOTLYJS@@</script>
<style>
  :root { --accent:#be0f05; --ink:#1e293b; --muted:#64748b; --line:#e2e8f0; --bg:#fff; --panel:#f8fafc; }
  @media (prefers-color-scheme: dark) {
    :root { --ink:#e2e8f0; --muted:#94a3b8; --line:#334155; --bg:#0f172a; --panel:#1e293b; }
  }
  * { box-sizing:border-box; }
  body { margin:0; font-family:system-ui,-apple-system,sans-serif; color:var(--ink); background:var(--bg); }
  header { padding:32px 24px 18px; text-align:center; border-bottom:1px solid var(--line); }
  .brand { display:inline-block; background:var(--accent); color:#fff; padding:8px 16px;
           border-radius:10px; font-weight:800; letter-spacing:-.3px; font-size:22px; }
  .sub { color:var(--muted); font-size:13px; margin-top:12px; }
  .modebar { margin-top:16px; }
  .mode-toggle { border:1px solid var(--line); background:var(--panel); color:var(--ink);
                 border-radius:8px; padding:7px 14px; font-size:13px; font-weight:600; cursor:pointer; }
  .mode-toggle:hover { border-color:var(--accent); color:var(--accent); }
  .steps { display:flex; flex-wrap:wrap; justify-content:center; gap:4px; padding:12px 24px;
           border-bottom:1px solid var(--line); background:var(--panel); overflow-x:auto; }
  .step { border:1px solid var(--line); background:var(--bg); color:var(--muted); cursor:pointer;
          min-width:30px; height:30px; padding:0 6px; border-radius:6px; font-size:12px; font-weight:600; }
  .step:hover { border-color:var(--accent); color:var(--accent); }
  .step.on { background:var(--accent); border-color:var(--accent); color:#fff; }
  main { max-width:1040px; margin:0 auto; padding:8px 24px 40px; }
  section { padding:24px 0; border-bottom:1px solid var(--line); }
  section.page[hidden] { display:none; }
  h2 { font-size:20px; margin:0 0 4px; }
  .note { color:var(--muted); font-size:13px; margin:0 0 14px; }
  .chart { width:100%; overflow-x:auto; }
  footer.nav { position:sticky; bottom:0; display:flex; align-items:center; gap:16px;
               padding:12px 24px; border-top:1px solid var(--line); background:var(--panel); }
  button.navbtn { background:var(--accent); color:#fff; border:0; border-radius:8px;
                  padding:9px 18px; font-weight:600; cursor:pointer; font-size:14px; }
  button.navbtn[disabled] { opacity:.35; cursor:default; }
  .where { font-size:13px; color:var(--muted); flex:1; }
  .where b { color:var(--ink); }
  .credit { text-align:center; color:var(--muted); font-size:12px; padding:24px; }
  /* one-page mode: reveal every section, drop the paged chrome */
  body.onepage .steps, body.onepage footer.nav { display:none; }
  body.onepage section.page[hidden] { display:block; }
</style></head>
<body>
<header>
  <div class="brand">TIP4PATLIBS &mdash; Antibiotic Resistance</div>
  <div class="sub">Patent landscape &middot; EPO PATSTAT Global &middot; landscape analyses after Riccardo Priore, Centro PATLIB, AREA Science Park</div>
  <div class="modebar"><button class="mode-toggle" id="modeToggle"></button></div>
</header>
<div class="steps" id="steps">@@STEPS@@</div>
<main>@@SECTIONS@@</main>
<footer class="nav" id="nav">
  <button class="navbtn" id="prev">&larr; Previous</button>
  <button class="navbtn" id="next">Next &rarr;</button>
  <div class="where"><b id="pos"></b> &nbsp;<span id="label"></span></div>
</footer>
<div class="credit">Generated on EPO TIP from PATSTAT PROD &middot; @@COUNT@@ analyses &middot; self-contained, no internet needed.</div>
<script>
(function () {
  var pages = Array.prototype.slice.call(document.querySelectorAll('.page'));
  var steps = Array.prototype.slice.call(document.querySelectorAll('.step'));
  var titles = steps.map(function (b) { return b.getAttribute('title'); });
  var cur = -1;
  var KEY = 'tip4patlibs.report.mode';

  function resizeIn(el) {
    if (!window.Plotly) return;
    el.querySelectorAll('.plotly-graph-div').forEach(function (d) { Plotly.Plots.resize(d); });
  }
  function show(i) {
    if (i < 0 || i >= pages.length) return;
    if (cur >= 0) { pages[cur].hidden = true; if (steps[cur]) steps[cur].classList.remove('on'); }
    cur = i;
    pages[i].hidden = false;
    if (steps[i]) steps[i].classList.add('on');
    resizeIn(pages[i]);
    document.getElementById('pos').textContent = (i + 1) + ' / ' + pages.length;
    document.getElementById('label').textContent = titles[i] || '';
    document.getElementById('prev').disabled = (i === 0);
    document.getElementById('next').disabled = (i === pages.length - 1);
    if (location.hash !== '#' + (i + 1)) history.replaceState(null, '', '#' + (i + 1));
  }
  function setMode(mode, remember) {
    var one = (mode === 'onepage');
    document.body.classList.toggle('onepage', one);
    document.getElementById('modeToggle').textContent = one ? '▤ Paged view' : '▦ One-page view';
    if (remember) { try { localStorage.setItem(KEY, mode); } catch (e) {} }
    if (one) pages.forEach(resizeIn);
    else show(cur < 0 ? 0 : cur);
  }

  document.getElementById('prev').onclick = function () { show(cur - 1); };
  document.getElementById('next').onclick = function () { show(cur + 1); };
  steps.forEach(function (b) { b.onclick = function () { show(+b.dataset.go); }; });
  document.addEventListener('keydown', function (e) {
    if (document.body.classList.contains('onepage')) return;
    if (e.target && /^(INPUT|TEXTAREA|SELECT)$/.test(e.target.tagName)) return;
    if (e.key === 'ArrowLeft') show(cur - 1);
    if (e.key === 'ArrowRight') show(cur + 1);
  });
  document.getElementById('modeToggle').onclick = function () {
    setMode(document.body.classList.contains('onepage') ? 'paged' : 'onepage', true);
  };

  var start = parseInt((location.hash || '#1').slice(1), 10);
  show(isNaN(start) || start < 1 || start > pages.length ? 0 : start - 1);
  var saved = null;
  try { saved = localStorage.getItem(KEY); } catch (e) {}
  setMode(saved === 'onepage' ? 'onepage' : 'paged', false);
})();
</script>
</body></html>"""

html = (TEMPLATE
        .replace("@@PLOTLYJS@@", get_plotlyjs())
        .replace("@@STEPS@@", "".join(steps))
        .replace("@@SECTIONS@@", "".join(sections))
        .replace("@@COUNT@@", str(len(entries))))

report_path.write_text(html, encoding="utf-8")
print(f"Wrote {report_path}  —  {report_path.stat().st_size / 1e6:.1f} MB, "
      f"{len(entries)} sections (paged default + one-page toggle).")

Wrote /home/jovyan/epo-tip4patlibs/6_patentreports/2_antibiotic_resistance_rebuild/4_report/antibiotic_resistance_report.html  —  4.8 MB, 13 sections (paged default + one-page toggle).


## Step 3 — Build the data workbook (one sheet per chart)

Every chart's underlying data goes into one Excel workbook, one sheet per chart (a network contributes two: nodes and edges). This is the "show me the numbers" companion — the report is auditable and a client can reuse the data without touching TIP.

In [11]:
import re
import pandas as pd

data_path = REPORT_DIR / "antibiotic_resistance_report_data.xlsx"

def sheet_name(base, used):
    name = re.sub(r"[\\/*?:\[\]]", " ", base)[:31].strip() or "sheet"
    candidate, i = name, 1
    while candidate in used:
        i += 1
        candidate = f"{name[:28]}_{i}"
    used.add(candidate)
    return candidate

used = set()
with pd.ExcelWriter(data_path, engine="openpyxl") as writer:
    for e in entries:
        multi = len(e["sheets"]) > 1
        for label, path in e["sheet_paths"].items():
            # for multi-table charts keep the label (nodes/edges) visible within Excel's 31-char limit
            base = f"{e['title'][:22]} ({label})" if multi else e["title"]
            pd.read_parquet(path).to_excel(writer, sheet_name=sheet_name(base, used), index=False)

print(f"Wrote {data_path}  —  {len(used)} sheet(s).")

Wrote /home/jovyan/epo-tip4patlibs/6_patentreports/2_antibiotic_resistance_rebuild/4_report/antibiotic_resistance_report_data.xlsx  —  14 sheet(s).


## Step 4 — Open the report

The report is self-contained HTML, so we open it with the course's shared `open_html` helper: it serves the file through jupyter-server-proxy and shows a red **Open** button (plus a download link). That is the reliable way to view an interactive HTML artifact inside TIP — a plain link or an iframe would be blocked by Jupyter's sandbox.

In [12]:
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists()), Path.cwd())
sys.path.insert(0, str(root / "1_startwithtip"))
from tip_tools import open_html

open_html(report_path, "the antibiotic-resistance report")

---
**Done.** The report and its data workbook are in `4_report/`. This is the MVP spine: dataset
&rarr; five analyses &rarr; one self-contained report that renders in TIP (paged **and** one-page),
plus a matching data workbook. Further analyses slot in by following the same
`report_kit.record` contract &mdash; the step bar and both view modes pick them up automatically.